In [1]:
import numpy as np
import brainpy.math as bm

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import seaborn as sns

import os
os.chdir("..")
from ratinabox.Environment import Environment
from ratinabox.Agent import Agent


%matplotlib inline
%load_ext autoreload
%autoreload 2

bm.set_dt(1.)  #simulation time stamp 1 ms

### 1, Generate a 'real' trajectory with RatInABox (George et al, 2024, eLife)

In [ ]:
np.random.seed(444) #for reproducibility

simulate_time = 4 #in seconds
Env_size = 1.0 #in meters

Env = Environment(params={
    "scale":Env_size, #in meters
    "dimensionality": "2D",
})

Ag = Agent(Env, params = {
    "dt": 0.001, #time step 1 ms
    "speed_mean":0.25, #m/s
    "speed_std": 0.,
    "speed_coherence_time": 10,
    "rotational_velocity_std": 40*np.pi/180,  #set to 120 or 360 will increase the tutoriocity of the trajectory
    })

while Ag.t < simulate_time:
    Ag.update()
    
time_steps = np.array(Ag.history["t"])  
position = Ag.history['pos']
position = np.array(position)

# compute direction
direction_xy = Ag.history["head_direction"]
direction_xy = np.array(direction_xy)
direction = np.arctan2(direction_xy[:,1], direction_xy[:,0])

# compute angular velocity (unwrap to remove jumps at -pi/pi)
direction_unwrapped = np.unwrap(direction)
ang_velocity = np.diff(direction_unwrapped) / Ag.dt
ang_velocity = np.insert(ang_velocity, 0, 0)

# compute moving speed
moving_speed = np.linalg.norm(Ag.history['vel'], axis=1)

#plot 
fig, axs = plt.subplots(1,3,figsize=(12,3),dpi=100, width_ratios=[1,2,2])

ax = axs[0]
ax.plot(position[:,0], position[:,1], lw=1, color='black')
ax.set_xlim(0, Env_size)
ax.set_ylim(0, Env_size)
#equal axis
ax.set_aspect('equal', adjustable='box')
ax.set_xticks([0, Env_size])
ax.set_yticks([0, Env_size])

ax = axs[1]
sns.despine(ax=axs[1])
ax.plot(time_steps, moving_speed, lw=1, color='#009FB9')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Speed (m/s)')

ax = axs[2]
sns.despine(ax=axs[2])
# find the jump points where the difference between two adjacent points is greater than pi
jumps = np.where(np.abs(np.diff(direction)) > np.pi)[0]
# set the jump points to NaN for plotting
direction_plot = direction.copy()
direction_plot[jumps + 1] = np.nan
ax.plot(time_steps, direction_plot, lw=1, color='#009FB9')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Direction (rad)')

plt.tight_layout()


In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.animation import FuncAnimation
from matplotlib.patches import FancyArrowPatch
from IPython.display import HTML, display


mpl.rcParams["animation.embed_limit"] = 100


# ============================================================
# Parameters
# ============================================================
n_step = 30

display_size = 1.0
n_tiles = 2
tile_size = display_size / n_tiles

# Ring attractor
n_neurons = 30
ring_bump_sigma = np.deg2rad(22)

# Torus
torus_bump_sigma = 0.42
torus_major_radius = 2.4
torus_minor_radius = 0.75
torus_surface_resolution = 60

simulation_dt = 0.001

animation_step = n_step * simulation_dt

animation_interval = (
    animation_step * 1000
)

animation_fps = int(
    round(1.0 / animation_step)
)

blue_color = "#00B7FF"

teal_color = np.array(
    [0.0, 183 / 255, 1.0, 1.0]
)


# ============================================================
# Prepare trajectory
# ============================================================
position_all = np.asarray(position)
position_all = position_all[:, :2]


# ============================================================
# Calculate movement direction
# ============================================================
motion_vector = np.empty_like(position_all)

motion_vector[:-1] = (
    position_all[1:] - position_all[:-1]
)

motion_vector[-1] = motion_vector[-2]

motion_speed = np.linalg.norm(
    motion_vector,
    axis=1,
)

for i in range(1, len(motion_vector)):
    if motion_speed[i] < 1e-12:
        motion_vector[i] = motion_vector[i - 1]

motion_direction_all = np.arctan2(
    motion_vector[:, 1],
    motion_vector[:, 0],
)


# ============================================================
# Calculate torus phase coordinates
# ============================================================
phase_all = (
    2.0
    * np.pi
    * np.mod(position_all, tile_size)
    / tile_size
)

phase_u_all = phase_all[:, 0]
phase_v_all = phase_all[:, 1]


# ============================================================
# Subsample animation frames
# ============================================================
sample_indices = np.arange(
    0,
    len(position_all),
    n_step,
)

sample_indices = np.unique(
    np.append(
        sample_indices,
        len(position_all) - 1,
    )
)

position_ani = position_all[sample_indices]
direction_ani = motion_direction_all[sample_indices]

phase_u_ani = phase_u_all[sample_indices]
phase_v_ani = phase_v_all[sample_indices]

n_frames = len(position_ani)

arrow_length = 0.20


# ============================================================
# Helper functions
# ============================================================
def circular_distance(angle1, angle2):
    return np.angle(
        np.exp(1j * (angle1 - angle2))
    )


# ============================================================
# Ring attractor functions
# ============================================================
neuron_angles = np.linspace(
    -np.pi,
    np.pi,
    n_neurons,
    endpoint=False,
)

neuron_x = np.cos(neuron_angles)
neuron_y = np.sin(neuron_angles)


def get_ring_bump_activity(current_direction):
    distance = circular_distance(
        neuron_angles,
        current_direction,
    )

    activity = np.exp(
        -0.5
        * (distance / ring_bump_sigma) ** 2
    )

    activity /= np.max(activity)

    return activity


def get_neuron_colors(activity):
    gray = np.array(
        [0.72, 0.75, 0.80, 1.0]
    )

    activity = activity[:, None]

    return (
        (1.0 - activity) * gray
        + activity * teal_color
    )


# ============================================================
# Torus functions
# ============================================================
R = torus_major_radius
r = torus_minor_radius


def torus_xyz(u, v, surface_offset=0.0):
    tube_radius = r + surface_offset

    radius = R + tube_radius * np.cos(v)

    x = radius * np.cos(u)
    y = radius * np.sin(u)
    z = tube_radius * np.sin(v)

    return x, y, z


# ============================================================
# Torus surface
# ============================================================
surface_n = torus_surface_resolution

u_vertices = np.linspace(
    0,
    2 * np.pi,
    surface_n + 1,
)

v_vertices = np.linspace(
    0,
    2 * np.pi,
    surface_n + 1,
)

U_vertices, V_vertices = np.meshgrid(
    u_vertices,
    v_vertices,
)

T_X, T_Y, T_Z = torus_xyz(
    U_vertices,
    V_vertices,
)


# ============================================================
# Torus dots
# ============================================================
torus_n_u = 30
torus_n_v = 15

dot_base_size = 20
dot_bump_size = 100

u_dots = np.linspace(
    0,
    2 * np.pi,
    torus_n_u,
    endpoint=False,
)

v_dots = np.linspace(
    0,
    2 * np.pi,
    torus_n_v,
    endpoint=False,
)

U_dots, V_dots = np.meshgrid(
    u_dots,
    v_dots,
)

U_dots_flat = U_dots.flatten()
V_dots_flat = V_dots.flatten()

torus_dot_x, torus_dot_y, torus_dot_z = torus_xyz(
    U_dots_flat,
    V_dots_flat,
    surface_offset=0.0,
)


# ============================================================
# Torus lighting
# ============================================================
torus_normals = np.column_stack(
    [
        np.cos(V_dots_flat) * np.cos(U_dots_flat),
        np.cos(V_dots_flat) * np.sin(U_dots_flat),
        np.sin(V_dots_flat),
    ]
)

light_direction = np.array(
    [-0.35, -0.45, 0.82]
)

light_direction /= np.linalg.norm(
    light_direction
)

surface_light = torus_normals @ light_direction

surface_light = np.clip(
    surface_light,
    0.0,
    1.0,
)


def get_torus_dot_activity(current_u, current_v):
    du = circular_distance(
        U_dots_flat,
        current_u,
    )

    dv = circular_distance(
        V_dots_flat,
        current_v,
    )

    activity = np.exp(
        -0.5
        * (
            (du / torus_bump_sigma) ** 2
            + (dv / torus_bump_sigma) ** 2
        )
    )

    activity /= np.max(activity)

    return activity


def get_torus_dot_colors(activity):
    activity = np.asarray(activity)[:, None]

    gray_rgb = np.array(
        [0.90, 0.92, 0.96]
    )

    gray_lit = (
        gray_rgb[None, :]
        * (
            0.90
            + 0.10 * surface_light[:, None]
        )
    )

    teal_lit = (
        teal_color[:3][None, :]
        * (
            0.90
            + 0.10 * surface_light[:, None]
        )
    )

    rgb_colors = (
        (1.0 - activity) * gray_lit
        + activity * teal_lit
    )

    alpha = np.full(
        (len(activity), 1),
        0.75,
    )

    return np.hstack(
        [
            rgb_colors,
            alpha,
        ]
    )


# ============================================================
# Initial values
# ============================================================
current_position = position_ani[0]
current_direction = direction_ani[0]

initial_ring_activity = (
    get_ring_bump_activity(
        current_direction
    )
)

initial_torus_activity = (
    get_torus_dot_activity(
        phase_u_ani[0],
        phase_v_ani[0],
    )
)

initial_torus_colors = (
    get_torus_dot_colors(
        initial_torus_activity
    )
)


# ============================================================
# Figure 1: trajectory only
# ============================================================
fig_traj = plt.figure(
    figsize=(4, 3),
    dpi=120,
    constrained_layout=True,
)

ax_traj = fig_traj.add_subplot(1, 1, 1)


realspace_trajectory, = ax_traj.plot(
    [position_all[0, 0]],
    [position_all[0, 1]],
    color="black",
    linewidth=1.5,
    zorder=1,
)

realspace_dot, = ax_traj.plot(
    [current_position[0]],
    [current_position[1]],
    marker="o",
    color=blue_color,
    markersize=10,
    markeredgecolor="black",
    markeredgewidth=1.5,
    zorder=5,
)

arrow_end = current_position + arrow_length * np.array(
    [
        np.cos(current_direction),
        np.sin(current_direction),
    ]
)

direction_arrow = FancyArrowPatch(
    posA=tuple(current_position),
    posB=tuple(arrow_end),
    arrowstyle="-|>",
    mutation_scale=18,
    linewidth=3.0,
    color=blue_color,
    zorder=6,
)

ax_traj.add_patch(direction_arrow)

ax_traj.set_xlim(
    0,
    display_size,
)

ax_traj.set_ylim(
    0,
    display_size,
)

ax_traj.set_xticks(
    [0, display_size]
)

ax_traj.set_yticks(
    [0, display_size]
)

ax_traj.set_aspect(
    "equal",
    adjustable="box",
)


# ============================================================
# Figure 2: ring above torus
# ============================================================
fig_state = plt.figure(
    figsize=(4, 6),
    dpi=120,
    constrained_layout=True,
)

grid_state = fig_state.add_gridspec(
    2,
    1,
    height_ratios=[1.0, 1.35],
)

ax_ring = fig_state.add_subplot(
    grid_state[0, 0]
)

ax_torus = fig_state.add_subplot(
    grid_state[1, 0],
    projection="3d",
)


# ============================================================
# Top panel: ring attractor
# ============================================================
circle_theta = np.linspace(
    0,
    2 * np.pi,
    500,
)

ax_ring.plot(
    np.cos(circle_theta),
    np.sin(circle_theta),
    color="lightgray",
    linewidth=2.0,
    zorder=1,
)

ring_neurons = ax_ring.scatter(
    neuron_x,
    neuron_y,
    s=55 + 180 * initial_ring_activity,
    c=get_neuron_colors(
        initial_ring_activity
    ),
    edgecolors="white",
    linewidths=0.8,
    zorder=3,
)

ring_direction_line, = ax_ring.plot(
    [
        0,
        np.cos(current_direction),
    ],
    [
        0,
        np.sin(current_direction),
    ],
    color=blue_color,
    linewidth=1.5,
    alpha=0.8,
    zorder=2,
)

ax_ring.set_xlim(
    -1.5,
    1.5,
)

ax_ring.set_ylim(
    -1.5,
    1.5,
)

ax_ring.set_aspect(
    "equal",
    adjustable="box",
)

ax_ring.set_title(
    "Ring attractor",
    y=0.92,
    pad=0,
)

ax_ring.axis("off")


# ============================================================
# Bottom panel: torus
# ============================================================
ax_torus.plot_surface(
    T_X,
    T_Y,
    T_Z,
    rstride=2,
    cstride=2,
    color="#D5DEE8",
    alpha=0.10,
    edgecolor="#AAB7C4",
    linewidth=0.25,
    shade=True,
    antialiased=True,
)

torus_dots = ax_torus.scatter(
    torus_dot_x,
    torus_dot_y,
    torus_dot_z,
    s=dot_base_size
    + dot_bump_size * initial_torus_activity,
    c=initial_torus_colors,
    edgecolors="none",
    depthshade=False,
)

initial_torus_x, initial_torus_y, initial_torus_z = (
    torus_xyz(
        phase_u_ani[0],
        phase_v_ani[0],
        surface_offset=0.06,
    )
)

torus_current_point = ax_torus.scatter(
    [initial_torus_x],
    [initial_torus_y],
    [initial_torus_z],
    s=150,
    c=[teal_color],
    edgecolors="black",
    linewidths=1.0,
    depthshade=True,
)


# ============================================================
# Torus trajectory
# ============================================================
trajectory_torus_x, trajectory_torus_y, trajectory_torus_z = (
    torus_xyz(
        phase_u_all,
        phase_v_all,
        surface_offset=0.035,
    )
)

torus_trajectory, = ax_torus.plot(
    [trajectory_torus_x[0]],
    [trajectory_torus_y[0]],
    [trajectory_torus_z[0]],
    color="black",
    linewidth=1.2,
    zorder=10,
)

ax_torus.set_title(
    "Toroidal attractor",
    y=0.92,
    pad=0,
)

ax_torus.set_xlim(
    -(R + r),
    R + r,
)

ax_torus.set_ylim(
    -(R + r),
    R + r,
)

ax_torus.set_zlim(
    -r * 1.2,
    r * 1.2,
)

ax_torus.set_box_aspect(
    (0.75, 1.0, 0.95)
)

ax_torus.view_init(
    elev=60,
    azim=0,
)

ax_torus.set_proj_type("persp")
ax_torus.set_axis_off()


# ============================================================
# Animation update: trajectory figure
# ============================================================
def update_trajectory(frame):
    current_position = position_ani[frame]
    current_direction = direction_ani[frame]

    current_data_index = sample_indices[frame]

    realspace_trajectory.set_data(
        position_all[
            :current_data_index + 1,
            0,
        ],
        position_all[
            :current_data_index + 1,
            1,
        ],
    )

    realspace_dot.set_data(
        [current_position[0]],
        [current_position[1]],
    )

    arrow_end = current_position + arrow_length * np.array(
        [
            np.cos(current_direction),
            np.sin(current_direction),
        ]
    )

    direction_arrow.set_positions(
        tuple(current_position),
        tuple(arrow_end),
    )

    current_data_index = sample_indices[frame]

    ax_traj.set_title(
        f"Animal trajectory "
        f"(t = {current_data_index * simulation_dt:.3f} s)"
    )

    return (
        realspace_trajectory,
        realspace_dot,
        direction_arrow,
    )


# ============================================================
# Animation update: ring and torus figure
# ============================================================
def update_state(frame):
    current_direction = direction_ani[frame]

    current_u = phase_u_ani[frame]
    current_v = phase_v_ani[frame]

    current_data_index = sample_indices[frame]

    # Ring activity
    ring_activity = (
        get_ring_bump_activity(
            current_direction
        )
    )

    ring_neurons.set_sizes(
        55 + 180 * ring_activity
    )

    ring_neurons.set_facecolors(
        get_neuron_colors(
            ring_activity
        )
    )

    ring_direction_line.set_data(
        [
            0,
            np.cos(current_direction),
        ],
        [
            0,
            np.sin(current_direction),
        ],
    )

    # Torus trajectory
    torus_trajectory.set_data(
        trajectory_torus_x[
            :current_data_index + 1
        ],
        trajectory_torus_y[
            :current_data_index + 1
        ],
    )

    torus_trajectory.set_3d_properties(
        trajectory_torus_z[
            :current_data_index + 1
        ]
    )

    # Torus bump
    torus_activity = (
        get_torus_dot_activity(
            current_u,
            current_v,
        )
    )

    torus_dots.set_sizes(
        dot_base_size
        + dot_bump_size * torus_activity
    )

    torus_dots.set_facecolors(
        get_torus_dot_colors(
            torus_activity
        )
    )

    # Moving point on torus
    current_torus_x, current_torus_y, current_torus_z = (
        torus_xyz(
            current_u,
            current_v,
            surface_offset=0.06,
        )
    )

    torus_current_point._offsets3d = (
        [current_torus_x],
        [current_torus_y],
        [current_torus_z],
    )

    return (
        ring_neurons,
        ring_direction_line,
        torus_trajectory,
        torus_dots,
        torus_current_point,
    )


# ============================================================
# Create two animations
# ============================================================
ani_trajectory = FuncAnimation(
    fig_traj,
    update_trajectory,
    frames=n_frames,
    interval=animation_interval,
    blit=False,
)

ani_state = FuncAnimation(
    fig_state,
    update_state,
    frames=n_frames,
    interval=animation_interval,
    blit=False,
)




# ============================================================
# Save GIF files
# ============================================================
output_dir = Path(
    "/home/zilong/Desktop/ThetaSweepModelling/figures"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

trajectory_gif_path = (
    output_dir / "trajectory_only.gif"
)

ring_torus_gif_path = (
    output_dir / "ring_torus_vertical.gif"
)

ani_trajectory.save(
    str(trajectory_gif_path),
    writer="pillow",
    fps=animation_fps,
)

ani_state.save(
    str(ring_torus_gif_path),
    writer="pillow",
    fps=animation_fps,
)


# ============================================================
# Display both animations in the notebook
# ============================================================
plt.close(fig_traj)
plt.close(fig_state)

display(
    HTML(
        ani_trajectory.to_jshtml()
    )
)

display(
    HTML(
        ani_state.to_jshtml()
    )
)

print(
    f"Saved trajectory GIF to:\n{trajectory_gif_path}"
)

print(
    f"Saved ring + torus GIF to:\n{ring_torus_gif_path}"
)

### 2, Generate a long trajectory 

In [ ]:
np.random.seed(444) #for reproducibility

simulate_time = 50 #in seconds
Env_size = 1.0 #in meters

Env = Environment(params={
    "scale":Env_size, #in meters
    "dimensionality": "2D",
})

Ag = Agent(Env, params = {
    "dt": 0.1, #time step 1 ms
    "speed_mean":0.5, #m/s
    "speed_std": 0.,
    "speed_coherence_time": 10,
    "rotational_velocity_std": 40*np.pi/180,  #set to 120 or 360 will increase the tutoriocity of the trajectory
    })

while Ag.t < simulate_time:
    Ag.update()
    
time_steps = np.array(Ag.history["t"])  
position = Ag.history['pos']
position = np.array(position)

# compute direction
direction_xy = Ag.history["head_direction"]
direction_xy = np.array(direction_xy)
direction = np.arctan2(direction_xy[:,1], direction_xy[:,0])

# compute angular velocity (unwrap to remove jumps at -pi/pi)
direction_unwrapped = np.unwrap(direction)
ang_velocity = np.diff(direction_unwrapped) / Ag.dt
ang_velocity = np.insert(ang_velocity, 0, 0)

# compute moving speed
moving_speed = np.linalg.norm(Ag.history['vel'], axis=1)

#plot 
fig, axs = plt.subplots(1,3,figsize=(12,3),dpi=100, width_ratios=[1,2,2])

ax = axs[0]
ax.plot(position[:,0], position[:,1], lw=1, color='black')
ax.set_xlim(0, Env_size)
ax.set_ylim(0, Env_size)
#equal axis
ax.set_aspect('equal', adjustable='box')
ax.set_xticks([0, Env_size])
ax.set_yticks([0, Env_size])

ax = axs[1]
sns.despine(ax=axs[1])
ax.plot(time_steps, moving_speed, lw=1, color='#009FB9')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Speed (m/s)')

ax = axs[2]
sns.despine(ax=axs[2])
# find the jump points where the difference between two adjacent points is greater than pi
jumps = np.where(np.abs(np.diff(direction)) > np.pi)[0]
# set the jump points to NaN for plotting
direction_plot = direction.copy()
direction_plot[jumps + 1] = np.nan
ax.plot(time_steps, direction_plot, lw=1, color='#009FB9')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Direction (rad)')

plt.tight_layout()


In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.animation import FuncAnimation
from matplotlib.patches import FancyArrowPatch
from IPython.display import HTML, display


mpl.rcParams["animation.embed_limit"] = 100


# ============================================================
# Run simulation
# ============================================================
np.random.seed(444)

simulate_time = 500
Env_size = 1.0

Env = Environment(
    params={
        "scale": Env_size,
        "dimensionality": "2D",
    }
)

Ag = Agent(
    Env,
    params={
        # Sampling interval: 0.1 seconds
        "dt": 0.1,

        # Real movement speed remains 0.25 m/s
        "speed_mean": 0.25,
        "speed_std": 0.0,
        "speed_coherence_time": 10,
        "rotational_velocity_std": 40 * np.pi / 180,
    }
)

while Ag.t < simulate_time:
    Ag.update()


# ============================================================
# Extract simulation data
# ============================================================
time_steps = np.asarray(
    Ag.history["t"]
)

position = np.asarray(
    Ag.history["pos"]
)

position = position[:, :2]

direction_xy = np.asarray(
    Ag.history["head_direction"]
)

direction = np.arctan2(
    direction_xy[:, 1],
    direction_xy[:, 0],
)

direction_unwrapped = np.unwrap(
    direction
)

ang_velocity = np.diff(
    direction_unwrapped
) / Ag.dt

ang_velocity = np.insert(
    ang_velocity,
    0,
    0,
)

moving_speed = np.linalg.norm(
    np.asarray(
        Ag.history["vel"]
    ),
    axis=1,
)


# ============================================================
# Parameters
# ============================================================
n_step = 10

display_size = Env_size

n_tiles = 2
tile_size = display_size / n_tiles

# Ring attractor
n_neurons = 30
ring_bump_sigma = np.deg2rad(22)

# Torus
torus_bump_sigma = 0.42
torus_major_radius = 2.4
torus_minor_radius = 0.75
torus_surface_resolution = 60

# Actual simulation sampling interval
simulation_dt = Ag.dt

# One animation frame represents this much simulated time
animation_step = (
    n_step
    * simulation_dt
)

# GIF plays 10 times faster than real simulated time
playback_speed = 10.0

animation_interval = (
    1000.0
    * animation_step
    / playback_speed
)

animation_fps = (
    playback_speed
    / animation_step
)

# Keep only the most recent 2 second on torus
trajectory_window_seconds = 2.0

trajectory_window_samples = max(
    1,
    int(
        round(
            trajectory_window_seconds
            / simulation_dt
        )
    ),
)

blue_color = "#00B7FF"

teal_color = np.array(
    [0.0, 183 / 255, 1.0, 1.0]
)


# ============================================================
# Subsample animation frames
# ============================================================
position_all = position

sample_indices = np.arange(
    0,
    len(position_all),
    n_step,
)

# Include the final position
sample_indices = np.unique(
    np.append(
        sample_indices,
        len(position_all) - 1,
    )
)

position_ani = position_all[
    sample_indices
]

direction_ani = direction[
    sample_indices
]

n_frames = len(
    position_ani
)

arrow_length = 0.20


# ============================================================
# Calculate torus phase coordinates
# ============================================================
phase_all = (
    2.0
    * np.pi
    * np.mod(
        position_all,
        tile_size,
    )
    / tile_size
)

phase_u_all = phase_all[:, 0]
phase_v_all = phase_all[:, 1]

phase_u_ani = phase_u_all[
    sample_indices
]

phase_v_ani = phase_v_all[
    sample_indices
]


# ============================================================
# Helper functions
# ============================================================
def circular_distance(angle1, angle2):
    return np.angle(
        np.exp(
            1j
            * (angle1 - angle2)
        )
    )


# ============================================================
# Ring attractor functions
# ============================================================
neuron_angles = np.linspace(
    -np.pi,
    np.pi,
    n_neurons,
    endpoint=False,
)

neuron_x = np.cos(
    neuron_angles
)

neuron_y = np.sin(
    neuron_angles
)


def get_ring_bump_activity(
    current_direction
):
    distance = circular_distance(
        neuron_angles,
        current_direction,
    )

    activity = np.exp(
        -0.5
        * (
            distance
            / ring_bump_sigma
        ) ** 2
    )

    activity /= np.max(
        activity
    )

    return activity


def get_neuron_colors(activity):
    gray = np.array(
        [0.72, 0.75, 0.80, 1.0]
    )

    activity = activity[:, None]

    return (
        (1.0 - activity)
        * gray
        + activity
        * teal_color
    )


# ============================================================
# Torus functions
# ============================================================
R = torus_major_radius
r = torus_minor_radius


def torus_xyz(
    u,
    v,
    surface_offset=0.0,
):
    tube_radius = (
        r
        + surface_offset
    )

    radius = (
        R
        + tube_radius
        * np.cos(v)
    )

    x = radius * np.cos(u)
    y = radius * np.sin(u)
    z = tube_radius * np.sin(v)

    return x, y, z


# ============================================================
# Torus surface
# ============================================================
surface_n = torus_surface_resolution

u_vertices = np.linspace(
    0,
    2 * np.pi,
    surface_n + 1,
)

v_vertices = np.linspace(
    0,
    2 * np.pi,
    surface_n + 1,
)

U_vertices, V_vertices = np.meshgrid(
    u_vertices,
    v_vertices,
)

T_X, T_Y, T_Z = torus_xyz(
    U_vertices,
    V_vertices,
)


# ============================================================
# Torus dots
# ============================================================
torus_n_u = 30
torus_n_v = 15

dot_base_size = 20
dot_bump_size = 100

u_dots = np.linspace(
    0,
    2 * np.pi,
    torus_n_u,
    endpoint=False,
)

v_dots = np.linspace(
    0,
    2 * np.pi,
    torus_n_v,
    endpoint=False,
)

U_dots, V_dots = np.meshgrid(
    u_dots,
    v_dots,
)

U_dots_flat = U_dots.flatten()
V_dots_flat = V_dots.flatten()

torus_dot_x, torus_dot_y, torus_dot_z = (
    torus_xyz(
        U_dots_flat,
        V_dots_flat,
        surface_offset=0.0,
    )
)


# ============================================================
# Torus lighting
# ============================================================
torus_normals = np.column_stack(
    [
        np.cos(V_dots_flat)
        * np.cos(U_dots_flat),

        np.cos(V_dots_flat)
        * np.sin(U_dots_flat),

        np.sin(V_dots_flat),
    ]
)

light_direction = np.array(
    [-0.35, -0.45, 0.82]
)

light_direction /= np.linalg.norm(
    light_direction
)

surface_light = (
    torus_normals
    @ light_direction
)

surface_light = np.clip(
    surface_light,
    0.0,
    1.0,
)


def get_torus_dot_activity(
    current_u,
    current_v,
):
    du = circular_distance(
        U_dots_flat,
        current_u,
    )

    dv = circular_distance(
        V_dots_flat,
        current_v,
    )

    activity = np.exp(
        -0.5
        * (
            (du / torus_bump_sigma) ** 2
            + (dv / torus_bump_sigma) ** 2
        )
    )

    activity /= np.max(
        activity
    )

    return activity


def get_torus_dot_colors(activity):
    activity = np.asarray(
        activity
    )[:, None]

    gray_rgb = np.array(
        [0.90, 0.92, 0.96]
    )

    gray_lit = (
        gray_rgb[None, :]
        * (
            0.90
            + 0.10
            * surface_light[:, None]
        )
    )

    teal_lit = (
        teal_color[:3][None, :]
        * (
            0.90
            + 0.10
            * surface_light[:, None]
        )
    )

    rgb_colors = (
        (1.0 - activity)
        * gray_lit
        + activity
        * teal_lit
    )

    alpha = np.full(
        (len(activity), 1),
        0.75,
    )

    return np.hstack(
        [
            rgb_colors,
            alpha,
        ]
    )


# ============================================================
# Initial values
# ============================================================
current_position = position_ani[0]
current_direction = direction_ani[0]

initial_ring_activity = (
    get_ring_bump_activity(
        current_direction
    )
)

initial_torus_activity = (
    get_torus_dot_activity(
        phase_u_ani[0],
        phase_v_ani[0],
    )
)

initial_torus_colors = (
    get_torus_dot_colors(
        initial_torus_activity
    )
)


# ============================================================
# Figure 1: trajectory only
# ============================================================
fig_traj = plt.figure(
    figsize=(4, 3),
    dpi=120,
    constrained_layout=True,
)

ax_traj = fig_traj.add_subplot(
    1,
    1,
    1,
)

realspace_trajectory, = ax_traj.plot(
    [position_all[0, 0]],
    [position_all[0, 1]],
    color="#263238",
    alpha=0.40,
    linewidth=1.5,
    zorder=1,
)

realspace_dot, = ax_traj.plot(
    [current_position[0]],
    [current_position[1]],
    marker="o",
    color=blue_color,
    markersize=10,
    markeredgecolor="black",
    markeredgewidth=1.5,
    zorder=5,
)

arrow_end = (
    current_position
    + arrow_length
    * np.array(
        [
            np.cos(current_direction),
            np.sin(current_direction),
        ]
    )
)

direction_arrow = FancyArrowPatch(
    posA=tuple(current_position),
    posB=tuple(arrow_end),
    arrowstyle="-|>",
    mutation_scale=18,
    linewidth=3.0,
    color=blue_color,
    zorder=6,
)

ax_traj.add_patch(
    direction_arrow
)

ax_traj.set_xlim(
    0,
    display_size,
)

ax_traj.set_ylim(
    0,
    display_size,
)

ax_traj.set_xticks(
    [0, display_size]
)

ax_traj.set_yticks(
    [0, display_size]
)

ax_traj.set_aspect(
    "equal",
    adjustable="box",
)


# ============================================================
# Figure 2: ring above torus
# ============================================================
fig_state = plt.figure(
    figsize=(4, 6),
    dpi=120,
    constrained_layout=True,
)

grid_state = fig_state.add_gridspec(
    2,
    1,
    height_ratios=[1.0, 1.35],
)

ax_ring = fig_state.add_subplot(
    grid_state[0, 0]
)

ax_torus = fig_state.add_subplot(
    grid_state[1, 0],
    projection="3d",
)


# ============================================================
# Ring attractor
# ============================================================
circle_theta = np.linspace(
    0,
    2 * np.pi,
    500,
)

ax_ring.plot(
    np.cos(circle_theta),
    np.sin(circle_theta),
    color="lightgray",
    linewidth=2.0,
    zorder=1,
)

ring_neurons = ax_ring.scatter(
    neuron_x,
    neuron_y,
    s=55
    + 180
    * initial_ring_activity,
    c=get_neuron_colors(
        initial_ring_activity
    ),
    edgecolors="white",
    linewidths=0.8,
    zorder=3,
)

ring_direction_line, = ax_ring.plot(
    [
        0,
        np.cos(current_direction),
    ],
    [
        0,
        np.sin(current_direction),
    ],
    color=blue_color,
    linewidth=1.5,
    alpha=0.8,
    zorder=2,
)

ax_ring.set_xlim(
    -1.5,
    1.5,
)

ax_ring.set_ylim(
    -1.5,
    1.5,
)

ax_ring.set_aspect(
    "equal",
    adjustable="box",
)

ax_ring.set_title(
    "Ring attractor",
    y=0.92,
    pad=0,
)

ax_ring.axis("off")


# ============================================================
# Torus surface and dots
# ============================================================
ax_torus.plot_surface(
    T_X,
    T_Y,
    T_Z,
    rstride=2,
    cstride=2,
    color="#D5DEE8",
    alpha=0.10,
    edgecolor="#AAB7C4",
    linewidth=0.25,
    shade=True,
    antialiased=True,
)

torus_dots = ax_torus.scatter(
    torus_dot_x,
    torus_dot_y,
    torus_dot_z,
    s=(
        dot_base_size
        + dot_bump_size
        * initial_torus_activity
    ),
    c=initial_torus_colors,
    edgecolors="none",
    depthshade=False,
)

initial_torus_x, initial_torus_y, initial_torus_z = (
    torus_xyz(
        phase_u_ani[0],
        phase_v_ani[0],
        surface_offset=0.06,
    )
)

torus_current_point = ax_torus.scatter(
    [initial_torus_x],
    [initial_torus_y],
    [initial_torus_z],
    s=150,
    c=[teal_color],
    edgecolors="black",
    linewidths=1.0,
    depthshade=True,
)


# ============================================================
# Torus trajectory
# ============================================================
trajectory_torus_x, trajectory_torus_y, trajectory_torus_z = (
    torus_xyz(
        phase_u_all,
        phase_v_all,
        surface_offset=0.035,
    )
)

torus_trajectory, = ax_torus.plot(
    [trajectory_torus_x[0]],
    [trajectory_torus_y[0]],
    [trajectory_torus_z[0]],
    color="#263238",
    alpha=0.40,
    linewidth=1.2,
    zorder=10,
)

ax_torus.set_title(
    "Toroidal attractor",
    y=0.92,
    pad=0,
)

ax_torus.set_xlim(
    -(R + r),
    R + r,
)

ax_torus.set_ylim(
    -(R + r),
    R + r,
)

ax_torus.set_zlim(
    -r * 1.2,
    r * 1.2,
)

ax_torus.set_box_aspect(
    (0.75, 1.0, 0.95)
)

ax_torus.view_init(
    elev=60,
    azim=0,
)

ax_torus.set_proj_type(
    "persp"
)

ax_torus.set_axis_off()


# ============================================================
# Update trajectory animation
# ============================================================
def update_trajectory(frame):
    current_position = position_ani[frame]
    current_direction = direction_ani[frame]

    current_data_index = sample_indices[frame]

    # First figure keeps the complete trajectory
    realspace_trajectory.set_data(
        position_all[
            :current_data_index + 1,
            0,
        ],
        position_all[
            :current_data_index + 1,
            1,
        ],
    )

    realspace_dot.set_data(
        [current_position[0]],
        [current_position[1]],
    )

    arrow_end = (
        current_position
        + arrow_length
        * np.array(
            [
                np.cos(current_direction),
                np.sin(current_direction),
            ]
        )
    )

    direction_arrow.set_positions(
        tuple(current_position),
        tuple(arrow_end),
    )

    ax_traj.set_title(
        f"Animal trajectory "
        f"(t = "
        f"{current_data_index * simulation_dt:.2f}"
        f" s)"
    )

    return (
        realspace_trajectory,
        realspace_dot,
        direction_arrow,
    )


# ============================================================
# Update ring + torus animation
# ============================================================
def update_state(frame):
    current_direction = direction_ani[frame]

    current_u = phase_u_ani[frame]
    current_v = phase_v_ani[frame]

    current_data_index = sample_indices[frame]

    # Ring activity
    ring_activity = (
        get_ring_bump_activity(
            current_direction
        )
    )

    ring_neurons.set_sizes(
        55
        + 180
        * ring_activity
    )

    ring_neurons.set_facecolors(
        get_neuron_colors(
            ring_activity
        )
    )

    ring_direction_line.set_data(
        [
            0,
            np.cos(current_direction),
        ],
        [
            0,
            np.sin(current_direction),
        ],
    )

    # Keep only the latest 1 second of torus trajectory
    window_start_index = max(
        0,
        current_data_index
        - trajectory_window_samples,
    )

    torus_trajectory.set_data(
        trajectory_torus_x[
            window_start_index:
            current_data_index + 1
        ],
        trajectory_torus_y[
            window_start_index:
            current_data_index + 1
        ],
    )

    torus_trajectory.set_3d_properties(
        trajectory_torus_z[
            window_start_index:
            current_data_index + 1
        ]
    )

    # Torus activity
    torus_activity = (
        get_torus_dot_activity(
            current_u,
            current_v,
        )
    )

    torus_dots.set_sizes(
        dot_base_size
        + dot_bump_size
        * torus_activity
    )

    torus_dots.set_facecolors(
        get_torus_dot_colors(
            torus_activity
        )
    )

    # Moving point on torus
    current_torus_x, current_torus_y, current_torus_z = (
        torus_xyz(
            current_u,
            current_v,
            surface_offset=0.06,
        )
    )

    torus_current_point._offsets3d = (
        [current_torus_x],
        [current_torus_y],
        [current_torus_z],
    )

    return (
        ring_neurons,
        ring_direction_line,
        torus_trajectory,
        torus_dots,
        torus_current_point,
    )


# ============================================================
# Create animations
# ============================================================
ani_trajectory = FuncAnimation(
    fig_traj,
    update_trajectory,
    frames=n_frames,
    interval=animation_interval,
    blit=False,
)

ani_state = FuncAnimation(
    fig_state,
    update_state,
    frames=n_frames,
    interval=animation_interval,
    blit=False,
)

# ============================================================
# Third animation:
# directional cells + real-space grid cells
# ============================================================

n_selected_cells = 1

spike_color = "#FE211D"

# ============================================================
# Randomly select one directional cell and one grid cell
# ============================================================
selection_rng = np.random.default_rng(444)

all_direction_cell_preferences = np.array(
    [
        0.0,
        2.0 * np.pi / 3.0,
        4.0 * np.pi / 3.0,
    ]
)

all_grid_cell_offsets = np.array(
    [
        [0.00, 0.00],
        [0.08, 0.05],
        [0.15, 0.12],
    ]
)

selected_direction_index = (
    selection_rng.integers(
        0,
        len(
            all_direction_cell_preferences
        ),
    )
)

selected_grid_index = (
    selection_rng.integers(
        0,
        len(
            all_grid_cell_offsets
        ),
    )
)

# Only one cell is used
n_selected_cells = 1

direction_cell_preferences = (
    all_direction_cell_preferences[
        [selected_direction_index]
    ]
)

grid_cell_offsets = (
    all_grid_cell_offsets[
        [selected_grid_index]
    ]
)

# One color is enough because only one cell is selected
selected_cell_colors = [
    "#E53935"
]

print(
    "Selected directional cell:",
    selected_direction_index + 1,
)

print(
    "Selected grid cell:",
    selected_grid_index + 1,
)

cell_tuning_sigma = ring_bump_sigma
grid_spacing = tile_size
grid_field_sigma = 0.045

# Poisson firing parameters
baseline_rate_hz = 0.05
maximum_rate_hz = 10.0

# Spike jitter
direction_spike_jitter = np.deg2rad(4.0)
direction_radius_jitter = 0.025
grid_spike_jitter = 0.012

spike_rng = np.random.default_rng(444)


# ============================================================
# Activity functions
# ============================================================
def get_direction_cell_activity(
    current_direction,
    preferred_direction,
):
    distance = circular_distance(
        current_direction,
        preferred_direction,
    )

    activity = np.exp(
        -0.5
        * (
            distance
            / cell_tuning_sigma
        ) ** 2
    )

    return float(activity)


def periodic_distance(
    value,
    offset,
    period,
):
    return (
        (
            value
            - offset
            + period / 2.0
        )
        % period
        - period / 2.0
    )


def get_real_grid_cell_activity(
    current_x,
    current_y,
    offset_x,
    offset_y,
):
    dx = periodic_distance(
        current_x,
        offset_x,
        grid_spacing,
    )

    dy = periodic_distance(
        current_y,
        offset_y,
        grid_spacing,
    )

    activity = np.exp(
        -0.5
        * (
            (dx / grid_field_sigma) ** 2
            + (dy / grid_field_sigma) ** 2
        )
    )

    return activity


# ============================================================
# Pre-generate Poisson spikes
# ============================================================
direction_spike_angles = []
direction_spike_radii = []
direction_spike_frames = []

grid_spike_x = []
grid_spike_y = []
grid_spike_frames = []


for cell_index in range(
    n_selected_cells
):

    direction_angles_for_cell = []
    direction_radii_for_cell = []
    direction_frames_for_cell = []

    grid_x_for_cell = []
    grid_y_for_cell = []
    grid_frames_for_cell = []

    preferred_direction = (
        direction_cell_preferences[
            cell_index
        ]
    )

    offset_x = (
        grid_cell_offsets[
            cell_index,
            0,
        ]
    )

    offset_y = (
        grid_cell_offsets[
            cell_index,
            1,
        ]
    )

    for frame in range(
        n_frames
    ):

        current_direction = (
            direction_ani[frame]
        )

        current_x = (
            position_ani[frame, 0]
        )

        current_y = (
            position_ani[frame, 1]
        )

        # ----------------------------------------------------
        # Directional cell
        # ----------------------------------------------------
        direction_activity = (
            get_direction_cell_activity(
                current_direction,
                preferred_direction,
            )
        )

        direction_rate = (
            baseline_rate_hz
            + maximum_rate_hz/4
            * direction_activity ** 2
        )

        n_direction_spikes = (
            spike_rng.poisson(
                direction_rate
                * animation_step
            )
        )

        if n_direction_spikes > 0:

            spike_angles = (
                current_direction
                + spike_rng.normal(
                    0.0,
                    direction_spike_jitter,
                    n_direction_spikes,
                )
            )

            spike_radii = (
                1.04
                + spike_rng.normal(
                    0.0,
                    direction_radius_jitter,
                    n_direction_spikes,
                )
            )

            spike_angles = np.mod(
                spike_angles,
                2.0 * np.pi,
            )

            spike_radii = np.clip(
                spike_radii,
                0.98,
                1.12,
            )

            direction_angles_for_cell.extend(
                spike_angles.tolist()
            )

            direction_radii_for_cell.extend(
                spike_radii.tolist()
            )

            direction_frames_for_cell.extend(
                [frame] * n_direction_spikes
            )

        # ----------------------------------------------------
        # Real-space grid cell
        # ----------------------------------------------------
        grid_activity = (
            get_real_grid_cell_activity(
                current_x,
                current_y,
                offset_x,
                offset_y,
            )
        )

        grid_rate = (
            baseline_rate_hz
            + maximum_rate_hz
            * grid_activity ** 2
        )

        n_grid_spikes = (
            spike_rng.poisson(
                grid_rate
                * animation_step
            )
        )

        if n_grid_spikes > 0:

            spike_x = (
                current_x
                + spike_rng.normal(
                    0.0,
                    grid_spike_jitter,
                    n_grid_spikes,
                )
            )

            spike_y = (
                current_y
                + spike_rng.normal(
                    0.0,
                    grid_spike_jitter,
                    n_grid_spikes,
                )
            )

            spike_x = np.clip(
                spike_x,
                0.0,
                Env_size,
            )

            spike_y = np.clip(
                spike_y,
                0.0,
                Env_size,
            )

            grid_x_for_cell.extend(
                spike_x.tolist()
            )

            grid_y_for_cell.extend(
                spike_y.tolist()
            )

            grid_frames_for_cell.extend(
                [frame] * n_grid_spikes
            )

    direction_spike_angles.append(
        np.asarray(
            direction_angles_for_cell
        )
    )

    direction_spike_radii.append(
        np.asarray(
            direction_radii_for_cell
        )
    )

    direction_spike_frames.append(
        np.asarray(
            direction_frames_for_cell
        )
    )

    grid_spike_x.append(
        np.asarray(
            grid_x_for_cell
        )
    )

    grid_spike_y.append(
        np.asarray(
            grid_y_for_cell
        )
    )

    grid_spike_frames.append(
        np.asarray(
            grid_frames_for_cell
        )
    )


# ============================================================
# Create figure
# ============================================================
fig_cells = plt.figure(
    figsize=(4, 6),
    dpi=120,
    constrained_layout=True,
)

cell_grid = fig_cells.add_gridspec(
    2,
    1,
    height_ratios=[1.0, 1.0],
    hspace=0.25,
)

direction_axes = []
grid_axes = []

direction_spike_artists = []
grid_spike_artists = []

direction_lines = []
grid_trajectories = []
grid_current_points = []
grid_direction_arrows = []
grid_arrow_length = 0.12


# ============================================================
# First row: directional cells
# ============================================================
theta_tuning = np.linspace(
    0.0,
    2.0 * np.pi,
    360,
)

for cell_index in range(
    n_selected_cells
):

    ax = fig_cells.add_subplot(
        cell_grid[0, cell_index],
        projection="polar",
    )

    direction_axes.append(ax)

    preferred_direction = (
        direction_cell_preferences[
            cell_index
        ]
    )

    tuning_distance = circular_distance(
        theta_tuning,
        preferred_direction,
    )

    tuning_curve = np.exp(
        -0.5
        * (
            tuning_distance
            / cell_tuning_sigma
        ) ** 2
    )

    direction_line, = ax.plot(
        [
            direction_ani[0],
            direction_ani[0],
        ],
        [
            0.0,
            1.0,
        ],
        color="#00B7FF",
        linewidth=1.8,
        alpha=0.95,
    )

    direction_lines.append(
        direction_line
    )

    spike_artist = ax.scatter(
        [],
        [],
        s=18,
        color=spike_color,
        alpha=0.85,
        edgecolors="none",
    )

    direction_spike_artists.append(
        spike_artist
    )

    ax.set_ylim(
        0.0,
        1.15,
    )

    ax.set_theta_zero_location(
        "E"
    )

    ax.set_theta_direction(
        1
    )

    # Only show four angular ticks
    ax.set_xticks(
        np.deg2rad(
            [
                0,
                90,
                180,
                270,
            ]
        )
    )

    ax.set_xticklabels(
        [
            "0°",
            "90°",
            "180°",
            "270°",
        ]
    )

    # Remove radial y-ticks
    ax.set_yticks([])

    ax.set_title(
        f"A directional cell "
    )


# ============================================================
# Second row: real-space grid cells
# ============================================================
for cell_index in range(
    n_selected_cells
):

    ax = fig_cells.add_subplot(
        cell_grid[1, cell_index]
    )

    grid_axes.append(ax)

    # Cumulative animal trajectory
    trajectory_line, = ax.plot(
        [
            position_ani[0, 0]
        ],
        [
            position_ani[0, 1]
        ],
        color="#9EA7AD",
        linewidth=1.6,
        alpha=0.9,
        zorder=3,
    )

    grid_trajectories.append(
        trajectory_line
    )

    # Current animal position
    current_point = ax.scatter(
        [
            position_ani[0, 0]
        ],
        [
            position_ani[0, 1]
        ],
        s=45,
        color="#263238",
        edgecolors="white",
        linewidths=0.8,
        zorder=5,
    )

    grid_current_points.append(
        current_point
    )

    # Current heading arrow
    grid_direction_arrow = FancyArrowPatch(
        posA=(
            position_ani[0, 0],
            position_ani[0, 1],
        ),
        posB=(
            position_ani[0, 0]
            + grid_arrow_length * np.cos(direction_ani[0]),
            position_ani[0, 1]
            + grid_arrow_length * np.sin(direction_ani[0]),
        ),
        arrowstyle="-|>",
        mutation_scale=12,
        linewidth=1.8,
        color="#00B7FF",
        zorder=7,
    )

    ax.add_patch(
        grid_direction_arrow
    )

    grid_direction_arrows.append(
        grid_direction_arrow
    )

    # Red spikes
    spike_artist = ax.scatter(
        [],
        [],
        s=18,
        color=spike_color,
        alpha=0.90,
        edgecolors="none",
        zorder=6,
    )

    grid_spike_artists.append(
        spike_artist
    )

    ax.set_xlim(
        0.0,
        Env_size,
    )

    ax.set_ylim(
        0.0,
        Env_size,
    )

    ax.set_xticks(
        [
            0.0,
            Env_size,
        ]
    )

    ax.set_yticks(
        [
            0.0,
            Env_size,
        ]
    )

    ax.set_aspect(
        "equal",
        adjustable="box",
    )

    ax.set_title(
        f"A grid cell "
    )



# ============================================================
# Update third animation
# ============================================================
def update_cells(frame):

    current_direction = (
        direction_ani[frame]
    )

    current_x = (
        position_ani[frame, 0]
    )

    current_y = (
        position_ani[frame, 1]
    )

    # --------------------------------------------------------
    # Directional cells
    # --------------------------------------------------------
    for cell_index in range(
        n_selected_cells
    ):

        preferred_direction = (
            direction_cell_preferences[
                cell_index
            ]
        )

        direction_activity = (
            get_direction_cell_activity(
                current_direction,
                preferred_direction,
            )
        )

        direction_lines[
            cell_index
        ].set_data(
            [
                current_direction,
                current_direction,
            ],
            [
                0.0,
                1,
            ],
        )

        event_frames = (
            direction_spike_frames[
                cell_index
            ]
        )

        visible = (
            event_frames <= frame
        )

        spike_angles = (
            direction_spike_angles[
                cell_index
            ][visible]
        )

        spike_radii = (
            direction_spike_radii[
                cell_index
            ][visible]
        )

        if len(spike_angles) > 0:
            direction_spike_artists[
                cell_index
            ].set_offsets(
                np.column_stack(
                    [
                        spike_angles,
                        spike_radii,
                    ]
                )
            )
        else:
            direction_spike_artists[
                cell_index
            ].set_offsets(
                np.empty(
                    (0, 2)
                )
            )

    # --------------------------------------------------------
    # Real-space grid cells
    # --------------------------------------------------------
    for cell_index in range(
        n_selected_cells
    ):

        grid_trajectories[
            cell_index
        ].set_data(
            position_ani[
                : frame + 1,
                0,
            ],
            position_ani[
                : frame + 1,
                1,
            ],
        )

        grid_current_points[
            cell_index
        ].set_offsets(
            [
                [
                    current_x,
                    current_y,
                ]
            ]
        )

        grid_direction_arrows[
            cell_index
        ].set_positions(
            (
                current_x,
                current_y,
            ),
            (
                current_x
                + grid_arrow_length * np.cos(current_direction),
                current_y
                + grid_arrow_length * np.sin(current_direction),
            ),
        )

        event_frames = (
            grid_spike_frames[
                cell_index
            ]
        )

        visible = (
            event_frames <= frame
        )

        spike_x = (
            grid_spike_x[
                cell_index
            ][visible]
        )

        spike_y = (
            grid_spike_y[
                cell_index
            ][visible]
        )

        if len(spike_x) > 0:
            grid_spike_artists[
                cell_index
            ].set_offsets(
                np.column_stack(
                    [
                        spike_x,
                        spike_y,
                    ]
                )
            )
        else:
            grid_spike_artists[
                cell_index
            ].set_offsets(
                np.empty(
                    (0, 2)
                )
            )

    return (
        direction_spike_artists
        + direction_lines
        + grid_spike_artists
        + grid_trajectories
        + grid_current_points
        + grid_direction_arrows
    )


# ============================================================
# Create and save animation
# ============================================================
ani_cells = FuncAnimation(
    fig_cells,
    update_cells,
    frames=n_frames,
    interval=animation_interval,
    blit=False,
)

output_dir = Path(
    "/home/zilong/Desktop/ThetaSweepModelling/figures"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

cells_gif_path = (
    output_dir
    / "directional_and_real_grid_cells.gif"
)

ani_cells.save(
    str(cells_gif_path),
    writer="pillow",
    fps=animation_fps,
)

print(
    f"Saved cell animation to:\n"
    f"{cells_gif_path}"
)

# Optional preview
display(
    HTML(
        ani_cells.to_jshtml(
            fps=animation_fps
        )
    )
)

# ============================================================
# Save GIF files
# ============================================================


output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

trajectory_gif_path = (
    output_dir
    / "trajectory_only_10x.gif"
)

ring_torus_gif_path = (
    output_dir
    / "ring_torus_vertical_10x.gif"
)

ani_trajectory.save(
    str(trajectory_gif_path),
    writer="pillow",
    fps=animation_fps,
)

ani_state.save(
    str(ring_torus_gif_path),
    writer="pillow",
    fps=animation_fps,
)


# ============================================================
# Optional notebook preview
# ============================================================
show_in_notebook = True

if show_in_notebook:
    display(
        HTML(
            ani_trajectory.to_jshtml(
                fps=animation_fps
            )
        )
    )

    display(
        HTML(
            ani_state.to_jshtml(
                fps=animation_fps
            )
        )
    )


plt.close(fig_traj)
plt.close(fig_state)
plt.close(fig_cells)

print(
    f"Saved trajectory GIF to:\n"
    f"{trajectory_gif_path}"
)

print(
    f"Saved ring + torus GIF to:\n"
    f"{ring_torus_gif_path}"
)